# SDS PySpark Tutorial — Part 2
## Joins, Aggregations, and Scalability — Without Graph Algorithms Yet

Part 1 taught the Spark mental model.

Part 2 now has **one job only**:

> Learn how to manipulate distributed Spark DataFrames confidently before applying Spark to PageRank, HITS, triangles, or other SDS algorithms.

The main pattern is:

\[
\boxed{\text{JOIN} \rightarrow \text{GROUP} \rightarrow \text{AGGREGATE}}
\]

But every example in this notebook uses ordinary tabular data such as users, transactions, departments, sales, and lookup tables.

You do **not** need to know graph theory or PageRank to complete this notebook.

---

## What Part 2 covers

- inner, left, semi, and anti joins;
- joins with different column names;
- aliases and ambiguous columns;
- `groupBy` and aggregation;
- duplicates, `distinct`, and `dropDuplicates`;
- `union`;
- sorting and top-k;
- self-joins using ordinary tabular data;
- join explosion and skew;
- shuffles;
- `explain()`;
- caching;
- partitions;
- broadcast joins;
- safe and unsafe `collect()`;
- common Spark debugging patterns.

## What Part 2 intentionally does **not** cover yet

- PageRank;
- HITS;
- graph cleaning;
- directed/undirected graphs;
- symmetrization;
- two-hop graph paths;
- triangle counting;
- connected components;
- modularity;
- SimRank;
- spectral graph methods.

Those belong in Part 3 **after** you study the corresponding SDS textbooks.

> **v3 revision:** Book 2 is now completely graph-free. All graph-specific examples and concepts have been moved out of this notebook. Section 7 keeps the simple money-transfer example because it teaches a generic DataFrame pattern without requiring PageRank knowledge.

In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T

spark = SparkSession.builder.appName("SDS-PySpark-Part2").getOrCreate()
print("Spark version:", spark.version)

# 1. Build two tiny tables before learning joins

A join is easier if you first understand the rows manually.

**Events**

| user_id | action |
|---|---|
| u1 | login |
| u1 | download |
| u2 | login |
| u3 | login |
| u4 | login |

**Users**

| user_id | role |
|---|---|
| u1 | analyst |
| u2 | engineer |
| u3 | analyst |
| u5 | manager |

Notice:

- `u4` appears only in Events.
- `u5` appears only in Users.
- `u1` has two event rows.

Those details determine the join output.

In [ ]:
events = spark.createDataFrame([
    ("u1", "login", 10.0),
    ("u1", "download", 50.0),
    ("u2", "login", 12.0),
    ("u3", "login", 9.0),
    ("u4", "login", 15.0),
], ["user_id", "action", "bytes_mb"])

users = spark.createDataFrame([
    ("u1", "analyst"),
    ("u2", "engineer"),
    ("u3", "analyst"),
    ("u5", "manager"),
], ["user_id", "role"])

events.show()
users.show()

# 2. Inner join — keep matches on both sides

Ask:

> Which event rows have a matching user row?

`u1`, `u2`, `u3` match.

`u4` has no user record, so its event disappears.

`u5` has no event, so it also does not appear.

In [ ]:
inner = events.join(users, "user_id", "inner")
inner.show()

## 2.1 Why one `u1` user row matches two event rows

The user table has one `u1` row.

The event table has two `u1` rows.

Therefore the join produces two `u1` output rows.

Joins operate on **matching rows**, not on abstract “unique users” unless your inputs are actually unique.

# 3. Left join — preserve every row on the left

A left join says:

> Keep all rows from the left table. Attach matching data from the right when it exists.

For `u4`, `role` becomes null rather than dropping the event.

In [ ]:
left = events.join(users, "user_id", "left")
left.show()

## 3.1 Why a left join matters in ordinary tabular work

Suppose `customers` contains every registered customer:

```text
u1
u2
u3
u4
```

Suppose `purchase_totals` contains only customers who actually bought something:

```text
u1 → 500
u3 → 200
```

If you use an **inner join**, customers `u2` and `u4` disappear because they have no matching purchase row.

But perhaps your report must show **every customer**, including customers with zero purchases.

Then use:

```python
customers.join(purchase_totals, "user_id", "left")
```

and replace missing totals with zero.

The lesson is:

> Use a **left join** when every row from the left table must survive, even if the right table has no match.

# 4. Semi join — existence test

A left-semi join keeps left rows that have a matching right key, but does not append the right columns.

Think:

> “Keep events generated by IDs that are present in this approved/selected table.”

In [ ]:
events.join(users, "user_id", "left_semi").show()

# 5. Anti join — find missing matches

A left-anti join does the opposite:

> Keep left rows whose key has **no match** on the right.

This is extremely useful for debugging.

In [ ]:
events.join(users, "user_id", "left_anti").show()

## Guided checkpoint — joins

Suppose:

```text
customers:       u1 u2 u3 u4
purchase_totals: u1 u3
```

Your report must contain **all four customers**.

Which join should you use from `customers` to `purchase_totals`?

A. inner  
B. left

**Answer:** B.

If missing purchase information means the customer spent zero, you can fill the missing total with `0.0`.

# 6. Joins with different column names and aliases

Real tables do not always use the same column name for the same identifier.

For example:

```text
orders.customer_id
customers.id
```

The matching rule is:

```text
orders.customer_id = customers.id
```

Aliases make this easier to read and prevent ambiguous-column errors after a join.

In [ ]:
orders = spark.createDataFrame([
    (1001, "u1", 50.0),
    (1002, "u2", 75.0),
    (1003, "u1", 20.0),
], ["order_id", "customer_id", "amount"])

customer_info = spark.createDataFrame([
    ("u1", "Alice", "Manila"),
    ("u2", "Bob", "Cebu"),
    ("u3", "Carol", "Davao"),
], ["id", "name", "city"])

joined_orders = (
    orders.alias("o")
    .join(
        customer_info.alias("c"),
        F.col("o.customer_id") == F.col("c.id"),
        "left"
    )
    .select(
        F.col("o.order_id").alias("order_id"),
        F.col("o.customer_id").alias("customer_id"),
        F.col("c.name").alias("customer_name"),
        F.col("c.city").alias("city"),
        F.col("o.amount").alias("amount"),
    )
)

joined_orders.show()

# 7. The most important Spark pattern: JOIN → GROUP → SUM

At this point, **forget PageRank**. We are not learning PageRank yet.

We are learning a much simpler Spark pattern that will later appear inside PageRank, HITS, triangle algorithms, and other SDS problems:

```text
attach a value to rows
        ↓
create one contribution per relationship
        ↓
group contributions by a key
        ↓
add them together
```

For now, the goal is only to understand **how Spark moves and combines values between tables**.

We will use an ordinary money-transfer example first.

## 7.1 Imagine people sending money

Suppose we have this relationship table:

| sender | receiver |
|---|---|
| Alice | Bob |
| Alice | Carol |
| Bob | Carol |
| Carol | Alice |

In Spark:

In [ ]:
transfers = spark.createDataFrame([
    ("Alice", "Bob"),
    ("Alice", "Carol"),
    ("Bob", "Carol"),
    ("Carol", "Alice"),
], ["sender", "receiver"])

transfers.show()

We also have another table containing one value for each person:

| person | money |
|---|---:|
| Alice | 100 |
| Bob | 60 |
| Carol | 40 |

Do **not** worry yet about why they are distributing money. We only care about the Spark operations.

In [ ]:
money = spark.createDataFrame([
    ("Alice", 100.0),
    ("Bob", 60.0),
    ("Carol", 40.0),
], ["person", "money"])

money.show()

## 7.2 The first problem: the relationship table does not contain the money

`transfers` tells us **who is connected to whom**, but not how much money each sender has.

The matching rule is:

```text
transfers.sender = money.person
```

That is exactly what a **join** can do: attach information from one table to matching rows in another.

In [ ]:
joined = (
    transfers.alias("t")
    .join(
        money.alias("m"),
        F.col("t.sender") == F.col("m.person"),
        "inner"
    )
    .select(
        F.col("t.sender").alias("sender"),
        F.col("t.receiver").alias("receiver"),
        F.col("m.money").alias("money")
    )
)

joined.show()

### Predict the output

You should expect:

```text
sender   receiver   money
Alice    Bob        100
Alice    Carol      100
Bob      Carol       60
Carol    Alice       40
```

Alice appears twice in the transfer table, so her value `100` is attached to both Alice rows.

> **A join can attach information about an entity to every matching relationship row.**

## 7.3 Suppose each sender splits the money equally

Alice sends to two people:

```text
Alice → Bob
Alice → Carol
```

So:

\[
100/2 = 50
\]

Bob has one receiver, so Bob sends 60.

Carol has one receiver, so Carol sends 40.

Before computing the amount per relationship, Spark first needs to know how many outgoing rows each sender has.

## 7.4 Count how many receivers each sender has

This is a `groupBy + count` problem.

Conceptually:

```text
Alice → [Bob, Carol] → 2
Bob   → [Carol]      → 1
Carol → [Alice]      → 1
```

In [ ]:
num_receivers = (
    transfers
    .groupBy("sender")
    .agg(F.count("*").alias("n_receivers"))
)

num_receivers.show()

Expected:

```text
sender   n_receivers
Alice    2
Bob      1
Carol    1
```

`groupBy("sender")` means:

> Put rows having the same sender into the same logical group.

Then `count("*")` counts the rows in each group.

## 7.5 Attach both pieces of information to each transfer

Each transfer now needs:

- the sender's money;
- the sender's number of receivers.

In [ ]:
prepared = (
    transfers.alias("t")
    .join(
        money.alias("m"),
        F.col("t.sender") == F.col("m.person"),
        "inner"
    )
    .join(
        num_receivers.alias("n"),
        F.col("t.sender") == F.col("n.sender"),
        "inner"
    )
    .select(
        F.col("t.sender").alias("sender"),
        F.col("t.receiver").alias("receiver"),
        F.col("m.money").alias("money"),
        F.col("n.n_receivers").alias("n_receivers")
    )
)

prepared.show()

Predict:

```text
sender   receiver   money   n_receivers
Alice    Bob        100     2
Alice    Carol      100     2
Bob      Carol       60     1
Carol    Alice       40     1
```

Now every relationship row contains the information needed to compute the value it carries.

## 7.6 Create one contribution per relationship

Compute:

```text
contribution = money / number of receivers
```

In [ ]:
contributions = (
    prepared
    .withColumn(
        "contribution",
        F.col("money") / F.col("n_receivers")
    )
)

contributions.select(
    "sender", "receiver", "contribution"
).show()

Expected:

```text
sender   receiver   contribution
Alice    Bob         50
Alice    Carol       50
Bob      Carol       60
Carol    Alice       40
```

Now each row means:

> **One value is travelling from one entity to another.**

That mental model becomes very useful in graph algorithms.

## 7.7 A receiver can receive several contributions

Carol receives two rows:

```text
Alice → Carol   50
Bob   → Carol   60
```

To compute Carol's total, put all rows with the same receiver together.

That is `groupBy("receiver")`.

In [ ]:
received = (
    contributions
    .groupBy("receiver")
    .agg(
        F.sum("contribution").alias("total_received")
    )
)

received.show()

Verify manually:

```text
Alice receives:
Carol → Alice = 40
TOTAL = 40

Bob receives:
Alice → Bob = 50
TOTAL = 50

Carol receives:
Alice → Carol = 50
Bob   → Carol = 60
TOTAL = 110
```

Expected Spark result:

```text
receiver   total_received
Alice       40
Bob         50
Carol      110
```

## 7.8 Remove the money story: the reusable Spark pattern

The general pattern is:

```text
RELATIONSHIP TABLE
sender → receiver

        +

ENTITY VALUE TABLE
entity → value

        ↓ JOIN

attach entity values
to relationship rows

        ↓

compute one contribution
per relationship

        ↓ GROUP BY receiver

put contributions for the
same receiver together

        ↓ SUM

one total per receiver
```

In compact form:

\[
\boxed{\text{JOIN} \rightarrow \text{CONTRIBUTION} \rightarrow \text{GROUP BY} \rightarrow \text{SUM}}
\]

This is one of the most important distributed-data patterns in the SDS exam.

## 7.9 Why are we learning this?

Later, the names may change.

Instead of:

```text
sender
receiver
money
contribution
```

you may see:

```text
source node
destination node
score
contribution
```

But the Spark mechanics can still be:

```text
join a value onto each relationship
        ↓
calculate what that relationship carries
        ↓
group by destination
        ↓
sum
```

This pattern appears inside PageRank and HITS.

**You do not need to understand PageRank yet.**

When you study the PageRank textbook, you will learn *why* those values are sent and what the score means. Book 2 is only teaching the Spark data-flow pattern.

## 7.10 Guided checkpoint

Given:

```text
sender   receiver   contribution
A        X          3
B        X          7
C        Y          4
```

After:

```python
.groupBy("receiver").agg(
    F.sum("contribution").alias("total_received")
)
```

first group:

```text
X → [3, 7]
Y → [4]
```

then sum:

```text
X → 10
Y → 4
```

That is the whole idea.

## 7.11 Independent checkpoint

Given:

```text
sender   receiver   contribution
A        C          2
B        C          5
A        D          2
E        D          8
```

Calculate manually:

1. total received by C;
2. total received by D.

Expected:

```text
C → 7
D → 10
```

Then express it in Spark:

```python
result = (
    contributions
    .groupBy("receiver")
    .agg(F.sum("contribution").alias("total_received"))
)
```

## 7.12 What you need to remember from Section 7

Only three ideas:

1. **JOIN attaches information to matching rows.**
2. **`groupBy` puts rows with the same key together.**
3. **Aggregation combines the values in each group.**

The complete pattern is:

```text
relationships
    +
entity values
    ↓
 JOIN
    ↓
one contribution per relationship
    ↓
 GROUP BY destination
    ↓
 SUM
```

If this makes sense, you have learned what Section 7 needs to teach.

**Do not stop here to master PageRank.** Save PageRank theory for the dedicated PageRank textbook and Part 3.

# 8. `groupBy + agg` in more detail

Suppose we have transactions:

| account | category | amount |
|---|---|---:|
| a | x | 10 |
| a | x | 15 |
| a | y | 20 |
| b | x | 8 |
| b | z | 12 |

Before running Spark, compute mentally:

- total amount for `a`;
- count of rows for `a`;
- average amount for `a`.

In [ ]:
tx = spark.createDataFrame([
    ("a", "x", 10.0),
    ("a", "x", 15.0),
    ("a", "y", 20.0),
    ("b", "x", 8.0),
    ("b", "z", 12.0),
], ["account", "category", "amount"])

tx.groupBy("account").agg(
    F.sum("amount").alias("total"),
    F.count("*").alias("n"),
    F.avg("amount").alias("avg"),
    F.max("amount").alias("max"),
).show()

## 8.1 Count rows versus count non-null values

`F.count("*")` counts rows.

`F.count("column")` counts non-null values in that column.

This difference matters when nulls exist.

In [ ]:
null_demo = spark.createDataFrame([
    ("a", 1.0),
    ("a", None),
    ("a", 3.0),
], ["key", "value"])

null_demo.groupBy("key").agg(
    F.count("*").alias("rows"),
    F.count("value").alias("non_null_values")
).show()

# 9. Duplicates: `distinct()` and `dropDuplicates()`

Duplicate rows are common in real data.

Suppose an import accidentally contains the same order twice:

In [ ]:
orders_with_duplicates = spark.createDataFrame([
    (1001, "u1", 50.0),
    (1001, "u1", 50.0),   # exact duplicate
    (1002, "u2", 75.0),
    (1003, "u1", 20.0),
], ["order_id", "customer_id", "amount"])

orders_with_duplicates.show()

## 9.1 `distinct()`

`distinct()` removes rows that are identical across **all columns**.

In [ ]:
orders_with_duplicates.distinct().show()

## 9.2 `dropDuplicates([...])`

Sometimes you decide uniqueness using only certain columns.

For example, if `order_id` should uniquely identify an order:

```python
df.dropDuplicates(["order_id"])
```

tells Spark to keep one row per `order_id`.

Be careful: if two rows share an `order_id` but disagree on other fields, you should first investigate whether the data itself is inconsistent.

In [ ]:
orders_with_duplicates.dropDuplicates(["order_id"]).show()

### Beginner checkpoint

What is the difference?

- `distinct()` → compare the **entire row**.
- `dropDuplicates(["order_id"])` → decide duplicates using only `order_id`.

# 10. `union`: stack rows from compatible DataFrames

A union places the rows of one DataFrame underneath the rows of another.

Suppose sales came from two files:

In [ ]:
sales_day1 = spark.createDataFrame([
    (1, "Manila", 100.0),
    (2, "Cebu", 80.0),
], ["sale_id", "branch", "amount"])

sales_day2 = spark.createDataFrame([
    (3, "Manila", 120.0),
    (4, "Davao", 90.0),
], ["sale_id", "branch", "amount"])

all_sales = sales_day1.union(sales_day2)
all_sales.show()

`union()` does **not** automatically remove duplicates.

If duplicate rows must be removed:

```python
df1.union(df2).distinct()
```

But only deduplicate when that matches the meaning of the data.

# 11. Sorting and top-k results

A common exam/reporting task is:

> Show the top 10 customers, products, branches, or scores.

For a small final result:

```python
df.orderBy(F.desc("amount")).limit(10)
```

is easy to read.

In [ ]:
all_sales.orderBy(F.desc("amount")).show()

If you deliberately reduce the result to a small number of rows, bringing that small result to Python is usually reasonable:

```python
top2 = (
    all_sales
    .orderBy(F.desc("amount"))
    .limit(2)
    .collect()
)
```

In [ ]:
top2 = (
    all_sales
    .orderBy(F.desc("amount"))
    .limit(2)
    .collect()
)

top2

# 12. Self-join: joining a table to itself

A **self-join** means using the same DataFrame twice.

We can learn this without graphs.

Suppose we have employees and departments:

In [ ]:
employees = spark.createDataFrame([
    ("e1", "Alice", "IT"),
    ("e2", "Bob", "IT"),
    ("e3", "Carol", "HR"),
    ("e4", "Dave", "HR"),
    ("e5", "Eve", "Finance"),
], ["employee_id", "name", "department"])

employees.show()

Question:

> Which pairs of **different employees** work in the same department?

Use two aliases of the same DataFrame:

```text
left employee.department = right employee.department
```

and require:

```text
left.employee_id < right.employee_id
```

to avoid self-pairs and reversed duplicates.

In [ ]:
same_department_pairs = (
    employees.alias("a")
    .join(
        employees.alias("b"),
        F.col("a.department") == F.col("b.department"),
        "inner"
    )
    .filter(F.col("a.employee_id") < F.col("b.employee_id"))
    .select(
        F.col("a.name").alias("employee_1"),
        F.col("b.name").alias("employee_2"),
        F.col("a.department").alias("department"),
    )
)

same_department_pairs.show()

Expected pairs:

```text
Alice - Bob   → IT
Carol - Dave  → HR
```

The important Spark lesson:

> A self-join compares/composes rows within the same table by treating the table as two logical copies.

# 13. Join explosion — why joins can create far more rows than you expect

Suppose one customer has:

- 100 purchase rows;
- 50 support-ticket rows.

If you join the two tables **only on customer ID**, every purchase for that customer can match every support ticket.

That can produce:

\[
100 \times 50 = 5{,}000
\]

rows for that customer alone.

This is called **join explosion**.

The issue is not that Spark is wrong. Spark is faithfully applying the matching rule you gave it.

## 13.1 Tiny demonstration

For one customer:

```text
purchases:
P1
P2
P3

tickets:
T1
T2
```

A join on customer alone creates:

```text
P1-T1
P1-T2
P2-T1
P2-T2
P3-T1
P3-T2
```

That is \(3\times2=6\) rows.

In [ ]:
purchases = spark.createDataFrame([
    ("u1", "P1"),
    ("u1", "P2"),
    ("u1", "P3"),
], ["customer_id", "purchase_id"])

tickets = spark.createDataFrame([
    ("u1", "T1"),
    ("u1", "T2"),
], ["customer_id", "ticket_id"])

purchases.join(tickets, "customer_id").show()

### Exam-useful question

If the join output is unexpectedly huge, ask:

1. Is the join key really unique on either side?
2. Did I intend a many-to-many join?
3. Should I aggregate/deduplicate one side before joining?

# 14. What is a shuffle?

Suppose sales rows are initially distributed like this:

```text
Executor 1: Manila, Cebu
Executor 2: Manila, Davao
Executor 3: Cebu, Manila
```

Now you ask:

```python
sales.groupBy("branch").sum("amount")
```

To add **all Manila sales together**, Spark may need to move rows between partitions so rows with the same branch can meet.

Conceptually:

```text
arbitrary distribution
        ↓
move data across partitions/workers
        ↓
Manila rows together
Cebu rows together
Davao rows together
        ↓
aggregate
```

That data movement is a **shuffle**.

Common operations that can involve shuffles include:

- joins;
- `groupBy`;
- `distinct`;
- global sorting;
- `repartition`.

A shuffle is not automatically a mistake. Many correct distributed algorithms need one.

The exam-useful idea is simply:

> **Shuffles can be expensive because data must move across the cluster.**

# 15. Lazy evaluation and `explain()`

You learned in Part 1 that Spark transformations are lazy.

`explain()` lets you inspect the planned computation.

In [ ]:
plan_demo = (
    all_sales
    .filter(F.col("amount") >= 90)
    .groupBy("branch")
    .agg(F.sum("amount").alias("total_sales"))
)

plan_demo.explain()

You do **not** need to decode every line of the physical plan for this exam.

For now, ask:

- Is there a join?
- Is there an aggregation?
- Is there sorting?
- Is data being exchanged/shuffled?
- Did I accidentally create a more complicated pipeline than intended?

# 16. Caching: why it exists

Suppose you create an expensive cleaned/aggregated DataFrame and reuse it many times.

Because Spark builds lazy lineages, repeated actions may otherwise cause earlier computation to be revisited.

`cache()` says:

> “I expect to reuse this DataFrame. Keep computed partitions available if possible.”

In [ ]:
branch_totals = (
    all_sales
    .groupBy("branch")
    .agg(F.sum("amount").alias("total_sales"))
    .cache()
)

# First action materializes the computation.
branch_totals.count()

branch_totals.orderBy(F.desc("total_sales")).show()
branch_totals.filter(F.col("total_sales") >= 100).show()

In [ ]:
branch_totals.unpersist()

## 16.1 Cache is not automatically good

Caching consumes resources.

Cache when:

- a costly DataFrame is reused;
- an iterative/repeated workflow keeps referring to it.

Do not cache everything merely because `.cache()` exists.

# 17. Partitions: only the intuition needed for the exam

A DataFrame is divided into partitions.

Very roughly:

- too few partitions can limit parallel work;
- too many tiny partitions add scheduling overhead;
- skew can make one partition much heavier than others.

Useful methods include:

```python
df.repartition(n)
df.coalesce(n)
```

For this exam, do not randomly repartition data. Use these only when you have a reason.

In [ ]:
print("Current partitions:", all_sales.rdd.getNumPartitions())

rep = all_sales.repartition(4)
print("After repartition(4):", rep.rdd.getNumPartitions())

coal = rep.coalesce(2)
print("After coalesce(2):", coal.rdd.getNumPartitions())

# 18. Broadcast joins — only when one side is genuinely small

Suppose you have:

- millions of transaction rows;
- a tiny table mapping 10 branch codes to branch names.

Instead of treating both sides like large distributed tables, Spark can distribute the tiny lookup table to workers.

Syntax:

```python
large_df.join(F.broadcast(tiny_lookup), "key")
```

The word **tiny** matters.

In [ ]:
branch_lookup = spark.createDataFrame([
    ("Manila", "Luzon"),
    ("Cebu", "Visayas"),
    ("Davao", "Mindanao"),
], ["branch", "region"])

sales_with_region = all_sales.join(
    F.broadcast(branch_lookup),
    "branch",
    "left"
)

sales_with_region.show()

Do not broadcast a table merely because you know the syntax.

Broadcast only data that is genuinely small enough to replicate safely.

# 19. Safe collection: use Spark as a funnel

A good distributed pattern is:

```text
huge distributed input
        ↓
filter / join / aggregate
        ↓
much smaller result
        ↓
limit / top-k / scalar
        ↓
collect
```

Usually reasonable:

```python
N = df.count()
top10 = df.orderBy(...).limit(10).collect()
small_summary = df.groupBy(...).agg(...).collect()
```

Potentially dangerous:

```python
all_rows = huge_df.collect()
```

when you have not established that `huge_df` is small.

# 20. Prefer Spark built-in functions to large Python row loops

Spark understands expressions such as:

```python
F.sum
F.count
F.avg
F.when
F.abs
F.sqrt
F.xxhash64
F.pmod
```

For large data, prefer expressing the computation through Spark DataFrame operations.

A weak scalable pattern is:

```python
rows = huge_df.collect()

for row in rows:
    # expensive Python processing
```

because the expensive state and computation have moved back to the driver.

The key exam question is:

> **Where does the expensive work happen?**

# 21. Common DataFrame debugging patterns

## 21.1 Unexpectedly fewer rows after a join

Ask:

- Did I use an inner join when I needed a left join?
- Are some keys missing from the right table?

## 21.2 Unexpectedly many rows after a join

Ask:

- Is the join many-to-many?
- Are there duplicate keys?
- Should one side be aggregated or deduplicated first?

## 21.3 Ambiguous column error

Use aliases and qualified references:

```python
F.col("a.id")
F.col("b.id")
```

## 21.4 Boolean filter error

Use:

```python
(condition1) & (condition2)
```

not Python `and` on Spark Columns.

## 21.5 Missing values after a left join

That may be correct. The right side simply had no match.

Use `fillna` only when the meaning of missing data really is a default such as zero.

## 21.6 Driver memory problem

Look for an unnecessary large:

```python
collect()
```

# 22. Part 2 practice

Answer these without graph knowledge.

### Q1
When should you prefer a left join over an inner join?

### Q2
What does a left-anti join return?

### Q3
Why are aliases useful?

### Q4
What is the difference between `distinct()` and `dropDuplicates(["id"])`?

### Q5
Does `union()` automatically remove duplicates?

### Q6
What is a self-join?

### Q7
A key occurs 20 times on the left and 30 times on the right. How many joined combinations can that key create?

### Q8
What is a shuffle in plain English?

### Q9
Why might `groupBy` cause a shuffle?

### Q10
When is caching useful?

### Q11
When is a broadcast join appropriate?

### Q12
Why is `.limit(10).collect()` usually safer than `.collect()` on a huge DataFrame?

### Q13
Your join output suddenly has 100 times more rows. What should you inspect first?

### Q14
What is the central Spark pattern taught in Section 7?

# 23. Part 2 answers

1. When all left-side rows must survive even without a match.
2. Left rows with **no** matching right-side key.
3. They make column origins explicit and prevent ambiguity.
4. `distinct()` compares entire rows; `dropDuplicates(["id"])` defines duplicates by selected key(s).
5. No.
6. Joining a DataFrame to another logical copy of itself.
7. \(20\times30=600\).
8. Data moves across partitions/workers so matching keys can be brought together.
9. Rows with the same group key may initially live in different partitions.
10. When a costly DataFrame is reused enough to justify persistence.
11. When one side is genuinely small enough to replicate safely.
12. It deliberately bounds the number of rows returned to the driver.
13. Whether the join keys are duplicated / many-to-many and whether the join condition is correct.
14. `JOIN → contribution → GROUP BY → SUM`.

# 24. Book 2 mental map

```text
NEED TO...                         FIRST SPARK THOUGHT

match rows                         join
keep only matches                  inner join
preserve every left row            left join
filter left by existence           left_semi
find missing matches               left_anti
same table used twice              self-join + aliases
summarize by key                   groupBy + agg
remove exact duplicate rows        distinct
dedupe by selected key             dropDuplicates
stack compatible rows              union
top-k                              orderBy + limit
reuse expensive DataFrame          cache
inspect plan                       explain
tiny lookup + huge table           broadcast tiny lookup
change partition count             repartition / coalesce
small final result to Python       collect may be fine
huge unbounded result to Python    avoid collect
```

The key questions to ask when reading Spark code are:

1. **What rows exist now?**
2. **What key is being used?**
3. **Is the join one-to-one, one-to-many, or many-to-many?**
4. **Which rows must be preserved?**
5. **What does each `groupBy` group together?**
6. **What does the aggregation compute?**
7. **Could this operation trigger a large shuffle or row explosion?**
8. **What is the smallest result that actually needs to reach the driver?**

# 25. Stop here before Book 3

If you can comfortably explain and modify the operations in Book 2, you have enough Spark mechanics to continue studying the SDS textbooks.

You do **not** need to know PageRank, HITS, triangles, or graph partitioning yet.

Recommended sequence:

```text
Book 1 — Spark mental model
        ↓
Book 2 — ordinary DataFrame mechanics
        ↓
SDS textbooks — learn the algorithms themselves
        ↓
Book 3 — apply Spark to those SDS algorithms
```

When you eventually reach Book 3, the graph code should feel like a new **application** of familiar Spark operations rather than a new programming language.